In [8]:
%load_ext autoreload
%autoreload 2

In [9]:
import os
os.environ['JAX_PLATFORMS'] = 'cpu'

In [25]:
import jax.numpy as jnp
import jax
import jax.random as random
from flax import linen as nn
import optax
from tqdm import tqdm
from utils import DataLoader
from utils import MLP

In [123]:
key = random.PRNGKey(0) # chiave per random 

f_to_learn = lambda mu, k, l, x: jnp.sin(2*mu*jnp.pi*x) + k + jnp.exp(-l*x)
N = 10000

key, subkey = random.split(key) # ogni volta, prima di usare la chiave, la devi dividere
x = random.uniform(subkey, (N,), minval=-10, maxval=10)
key, subkey = random.split(key)
mu = random.uniform(subkey, (N,), minval=-2, maxval=2)
key, subkey = random.split(key)
k = random.uniform(subkey, (N,), minval=-5, maxval=5)
key, subkey = random.split(key)
l = random.uniform(subkey, (N,), minval=-1, maxval=1)

y = f_to_learn(mu, k, l, x) # così generiamo artificialmente un dataset di N punti

In [125]:
X = jnp.stack([mu, k, l, x], axis=1)


split_idx = int(N * 0.8)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

train_dataloader = DataLoader(X_train, y_train, batch_size=32, shuffle=True)
test_dataloader = DataLoader(X_test, y_test, batch_size=32, shuffle=False)


In [9]:
# Example of iterating through the DataLoader
for data, label in train_dataloader:
    print(data.shape, label.shape)
    break

(32, 4) (32, 1)


In [10]:
targetnetwork = MLP(output_dim=1, hidden_dim=8, num_hidden_layers=1)

In [ ]:
x = jnp.ones((1,1)) #Gli input sono SEMPRE (SEMPRE) nel formato (bathc_size, input_dim1, input_dim2, ..., input_dimN)
# In questo caso, batch_size=1, input_dim=1
key = jax.random.key(0) # Bisogna sempre passare una key per inizializzare i pesi random
print(targetnetwork.tabulate(key, x)) # Visualizza la struttura del modello, con i pesi inizializzati


                               MLP Summary                               
┏━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┓
┃ path    ┃ module ┃ inputs       ┃ outputs      ┃ params               ┃
┡━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━┩
│         │ MLP    │ float32[1,1] │ float32[1,1] │                      │
├─────────┼────────┼──────────────┼──────────────┼──────────────────────┤
│ Dense_0 │ Dense  │ float32[1,1] │ float32[1,8] │ bias: float32[8]     │
│         │        │              │              │ kernel: float32[1,8] │
│         │        │              │              │                      │
│         │        │              │              │ 16 (64 B)            │
├─────────┼────────┼──────────────┼──────────────┼──────────────────────┤
│ Dense_1 │ Dense  │ float32[1,8] │ float32[1,1] │ bias: float32[1]     │
│         │        │              │              │ kernel: float32[8,1] │
│         │        │              │  

In [12]:
hypernetwork = MLP(output_dim = 25, hidden_dim=8, num_hidden_layers=2) # 25 come i parametri del target network

## First try: only training a single network

In [ ]:
key = jax.random.key(0)
key, subkey = random.split(key)

toy_data = random.uniform(key, (1000, 1), minval=-3, maxval=3)
toy_label = f_to_learn(0.5, 1.0, 0.1, toy_data)
train_dataloader = DataLoader(toy_data, toy_label, batch_size=32, shuffle=True)
test_data = random.uniform(key, (100, 1), minval=-3, maxval=3)
test_label = f_to_learn(0.5, 1.0, 0.1, test_data)
test_dataloader = DataLoader(test_data, test_label, batch_size=32, shuffle=False)

In [22]:
model = MLP(output_dim=1, hidden_dim=8, num_hidden_layers=1)

In [ ]:
def mse_loss(preds, targets):
    return jnp.mean((preds - targets) ** 2)

optimizer = optax.adam(learning_rate=1e-3)
params = model.init(jax.random.key(0), jnp.zeros((1, 1)))
opt_state = optimizer.init(params)
epochs = 1000

In [ ]:
# HO CAPITO GLI ITERATORI LOL
it = iter(train_dataloader)
next(it)

In [ ]:
def train_step(model, params, opt_state, loss, optimizer, x, y, key=None):
    loss_fn = lambda p, x, y: loss(model.apply(p, x), y)
    loss, grad = jax.value_and_grad(loss_fn)(params, x, y)
    
    updates, opt_state = optimizer.update(grad, opt_state, params)
    params = optax.apply_updates(params, updates)

    return params, opt_state, loss

train_step = jax.jit(train_step, static_argnames=('model', 'loss', 'optimizer'))

for epoch in tqdm(range(epochs)):
    epoch_loss = 0.0
    for data, label in train_dataloader:
        params, opt_state, loss = train_step(model = model, params = params, opt_state = opt_state, loss = mse_loss, optimizer = optimizer, x = data, y = label)
        epoch_loss += loss*len(data)
    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Training Loss: {epoch_loss / len(train_dataloader.data)}")
        test_loss = 0.0
        for data, label in test_dataloader:
            preds = model.apply(params, data)
            test_loss += mse_loss(preds, label) * len(data)
        print(f"Test Loss: {test_loss/ len(test_dataloader.data)}")

print(f"Final loss, training: {epoch_loss / len(train_dataloader.data)}, test: {test_loss / len(test_dataloader.data)}")

# FUNZIONA!

In [25]:
preds = model.apply(params, test_data)

https://huggingface.co/blog/afmck/flax-tutorial

https://wandb.ai/jax-series/simple-training-loop/reports/Writing-a-Training-Loop-in-JAX-and-Flax--VmlldzoyMzA4ODEy

## TRYING WITH NNX

In [140]:
from flax import nnx
import jax.random as random
import jax
import jax.numpy as jnp
from flax.nnx.training.metrics import Metric, Average
import os
os.environ['JAX_PLATFORMS'] = 'cpu'

In [141]:
f_to_learn = lambda mu, k, l, x: jnp.sin(2*mu*jnp.pi*x) + k + jnp.exp(-l*x)

key = jax.random.key(0)
key, subkey = random.split(key)
toy_data = random.uniform(subkey, (1000, 1), minval=-3, maxval=3)
toy_label = f_to_learn(0.5, 1.0, 0.1, toy_data)
train_dataloader = DataLoader(toy_data, toy_label, batch_size=32, shuffle=True)
key, subkey = random.split(key)
test_data = random.uniform(subkey, (100, 1), minval=-3, maxval=3)
test_label = f_to_learn(0.5, 1.0, 0.1, test_data)
test_dataloader = DataLoader(test_data, test_label, batch_size=32, shuffle=False)

In [227]:
class MLP(nnx.Module):
    input_dim: int
    output_dim: int = 1
    hidden_dim: int = 8
    num_hidden_layers: int = 1

    def __init__(self, input_dim: int, output_dim: int, hidden_dim: int, num_hidden_layers: int, *, rngs: nnx.Rngs):
        self.input_dim = input_dim
        self.output_dim = output_dim
        self.hidden_dim = hidden_dim
        self.num_hidden_layers = num_hidden_layers
        if num_hidden_layers > 0:
            self.hidden_layers = [nnx.Linear(self.input_dim, self.hidden_dim, rngs=rngs)] + \
                [nnx.Linear(self.hidden_dim, self.hidden_dim, rngs=rngs) for _ in range(self.num_hidden_layers - 1)]
            self.linear = nnx.Linear(hidden_dim, output_dim, rngs=rngs)
        else:
            self.linear = nnx.Linear(self.input_dim, self.output_dim, rngs=rngs)

    def __call__(self, x: jax.Array):
        if self.num_hidden_layers > 0:
            for i in range(self.num_hidden_layers):
                x = self.hidden_layers[i](x)
                x = nnx.relu(x) 
            x = self.linear(x) 
        else:
            x = self.linear(x)
        return x

In [228]:
model = MLP(input_dim = 1, output_dim = 1, hidden_dim = 8, num_hidden_layers = 2, rngs=nnx.Rngs(0))

In [229]:
def mse_loss(model: nnx.Module, x: jnp.ndarray, y: jnp.ndarray) -> jnp.ndarray:
    preds = model(x)
    return jnp.mean(optax.l2_loss(preds, y))

In [230]:
optimizer = nnx.Optimizer(model, optax.adam(learning_rate=1e-3))

In [231]:
training_loss = Average(argname='loss')
test_loss = Average(argname='loss')

def train_step(model: nnx.Module, loss_fn: callable, optimizer: nnx.Optimizer, metric: Metric, x: jax.Array, y: jax.Array):
    grad_fn = nnx.value_and_grad(loss_fn, has_aux=False)
    loss, grads = grad_fn(model, x, y)
    optimizer.update(grads)
    metric.update(loss=loss)

def evaluation_step(model: nnx.Module, loss_fn: callable, metric: Metric, x: jax.Array, y: jax.Array):
    loss = loss_fn(model, x, y)
    metric.update(loss=loss)

train_step = nnx.jit(train_step, static_argnames=('loss_fn'))
evaluation_step = nnx.jit(evaluation_step, static_argnames=('loss_fn'))

In [232]:
epochs = 1000
pbar = tqdm(range(epochs))
for epoch in pbar:
    pbar.set_description(f"Epoch {epoch+1}")
    training_loss.reset()
    test_loss.reset()

    for data, label in train_dataloader:
        train_step(model, mse_loss, optimizer, training_loss, data, label)

    for data, label in test_dataloader:
        evaluation_step(model, mse_loss, test_loss, data, label)
        
    pbar.set_postfix({"training loss": training_loss.compute(), "test loss": test_loss.compute()})

Epoch 1000: 100%|██████████| 1000/1000 [01:29<00:00, 11.21it/s, training loss=0.04300271, test loss=0.025178347]


## HYPERNETWORK

In [ ]:
# DEFINIZIONE DEL TRAINING ED EVALUATION STEP
def hypernetwork_train_step(targetnetwork: nnx.Module, hypernetwork: nnx.Module, loss_fn: callable, optimizer: nnx.Optimizer, metric: Metric, input_params: jax.Array, x: jax.Array, y: jax.Array, param_setter: callable): 
    def composition(input_params, x):
        parameters = hypernetwork(input_params)
        param_setter(targetnetwork, parameters)
        return targetnetwork(x)
    
    grad_fn = nnx.value_and_grad(loss_fn, has_aux=False)
    loss, grads = grad_fn(composition, x, y)
    optimizer.update(grads)
    metric.update(loss=loss)

def hypernetwork_evaluation_step(targetnetwork: nnx.Module, hypernetwork: nnx.Module, loss_fn: callable, metric: Metric, input_params: jax.Array, x: jax.Array, y: jax.Array, param_setter: callable):
    def composition(input_params, x):
        parameters = hypernetwork(input_params)
        param_setter(targetnetwork, parameters)
        return targetnetwork(x)
    
    loss = loss_fn(composition, x, y)
    metric.update(loss=loss)


hypernetwork_train_step = nnx.jit(hypernetwork_train_step, static_argnames=('loss_fn', 'param_setter'))
hypernetwork_evaluation_step = nnx.jit(hypernetwork_evaluation_step, static_argnames=('loss_fn', 'param_setter'))

In [263]:
# INIZIALIZZAZIONE DEL MODELLO
key = nnx.Rngs(0)
targetnetwork = MLP(input_dim = 1, output_dim = 1, hidden_dim = 8, num_hidden_layers = 1, rngs=key)
hypernetwork = MLP(input_dim = 1, output_dim = 25, hidden_dim = 8, num_hidden_layers = 2, rngs=key)
def set_params(model: nnx.Module, params: jax.Array):
    model.hidden_layers[0].kernel = params[:8]
    model.hidden_layers[0].bias = params[8:16]
    model.linear.kernel = params[16:24]
    model.linear.bias = params[24:]
def mse_loss(model: nnx.Module, x: jnp.ndarray, y: jnp.ndarray) -> jnp.ndarray:
    preds = model(x)
    return jnp.mean(optax.l2_loss(preds, y))
optimizer = nnx.Optimizer(hypernetwork, optax.adam(learning_rate=1e-3))
training_loss = Average(argname='loss')
test_loss = Average(argname='loss')

In [264]:
#GENERAZIONE DEL DATASET
f_to_learn = lambda mu, k, l, x: jnp.sin(2*mu*jnp.pi*x) + k + jnp.exp(-l*x)
N = 10000
x = random.uniform(key.params(), (N,), minval=-10, maxval=10)
mu = random.uniform(key.params(), (N,), minval=-2, maxval=2)
k = random.uniform(key.params(), (N,), minval=-5, maxval=5)
l = random.uniform(key.params(), (N,), minval=-0.2, maxval=0.2)
y = f_to_learn(mu, k, l, x) # così generiamo artificialmente un dataset di N punti
X = jnp.stack([mu, k, l, x], axis=1)

# DIVISIONE DEL DATASET IN TRAIN E TEST E CREAZIONE DEI DATALOADER
split_idx = int(N * 0.8)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]
train_dataloader = DataLoader(X_train, y_train, batch_size=32, shuffle=True)
test_dataloader = DataLoader(X_test, y_test, batch_size=32, shuffle=False)

In [265]:
epochs = 1000
pbar = tqdm(range(epochs))
for epoch in pbar:
    pbar.set_description(f"Epoch {epoch+1}")
    training_loss.reset()
    test_loss.reset()

    for data, label in train_dataloader:
        input_params = data[:, :-1] # mu, k, l
        x = data[:, -1:] # x
        hypernetwork_train_step(targetnetwork, hypernetwork, mse_loss, optimizer, training_loss, input_params, x, label, set_params)

    for data, label in test_dataloader:
        input_params = data[:, :-1] # mu, k, l
        x = data[:, -1:] # x
        hypernetwork_train_step(targetnetwork, hypernetwork, mse_loss, training_loss, input_params, x, label, set_params)
        
    pbar.set_postfix({"training loss": training_loss.compute(), "test loss": test_loss.compute()})

Epoch 1:   0%|          | 0/1000 [00:00<?, ?it/s]

Epoch 1:   0%|          | 0/1000 [00:00<?, ?it/s]


TypeError: Argument '<function hypernetwork_train_step.<locals>.composition at 0x717715a54680>' of type <class 'function'> is not a valid JAX type.

RIP NON POSSIAMO CALCOLARE LA LOSS SU COMPOSITION